### Residue Interaction Network Analysis of Proteins

Original tutorial: [Jonathan Zajac](https://github.com/Jonathan-Zajac/Tutorial-RIN). AAV8 example: SAMPEL Group.

Download this notebook and run the setup cell. Data files are fetched from GitHub when needed and cached locally; no login is required.
Use your existing Python environment; no packages are installed automatically.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import shutil
import tempfile

DATA_REPOSITORY = "SAMPEL-Group/RIN-framework"
DATA_COMMIT = "eb7397b1aca8c9b190aad92322f4e0aec3a5048d"
DATA_URL = f"https://raw.githubusercontent.com/{DATA_REPOSITORY}/{DATA_COMMIT}/tutorial"
DATA_CACHE = Path.cwd() / ".rin_tutorial_data" / DATA_COMMIT[:12]


def tutorial_data(relative):
    """Reuse an existing file, or download it from GitHub."""
    if Path(relative).is_absolute() or ".." in Path(relative).parts:
        raise ValueError("Use a path relative to the tutorial folder")
    for root in [Path.cwd(), Path.cwd() / "tutorial", Path.cwd().parent / "tutorial", DATA_CACHE]:
        path = root / relative
        if path.is_file():
            return path
    target = DATA_CACHE / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        print(f"Downloading {relative}…")
        with urlopen(f"{DATA_URL}/{relative}", timeout=45) as response:
            with tempfile.NamedTemporaryFile(dir=target.parent, delete=False) as output:
                temporary = Path(output.name)
                shutil.copyfileobj(response, output)
        temporary.replace(target)
        return target
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


import numpy as np
import networkx as nx
import MDAnalysis as md
from scipy.spatial import cKDTree


def heavy_atom_rin(protein, cutoff=6.0, sequence_exclusion=3, subunit_id=0):
    """One VP/protein: minimum heavy-atom distance < cutoff, no PBC."""
    if not np.isfinite(cutoff) or cutoff <= 0:
        raise ValueError("cutoff must be a positive distance in angstrom")
    if not isinstance(sequence_exclusion, int) or sequence_exclusion < 0:
        raise ValueError("sequence_exclusion must be a nonnegative integer")
    residues = protein.residues
    if len(set(residues.resids)) != len(residues):
        raise ValueError("Select one VP/protein before constructing this graph")
    heavy = protein.select_atoms("not name H*")
    xyz = heavy.positions.astype(np.float64)
    if not np.isfinite(xyz).all():
        raise ValueError("Nonfinite heavy-atom coordinates")
    atom_to_node = np.searchsorted(residues.ix, heavy.resindices)
    if len(np.unique(atom_to_node)) != len(residues):
        raise ValueError("Every residue must contain a heavy atom")
    atom_pairs = cKDTree(xyz).query_pairs(cutoff, output_type="ndarray")
    # The spatial query includes the boundary; the tutorial uses strict <.
    delta = xyz[atom_pairs[:, 0]] - xyz[atom_pairs[:, 1]]
    atom_pairs = atom_pairs[np.einsum("ij,ij->i", delta, delta) < cutoff**2]
    residue_pairs = np.sort(atom_to_node[atom_pairs], axis=1)
    residue_pairs = residue_pairs[
        residue_pairs[:, 1] - residue_pairs[:, 0] > sequence_exclusion
    ]
    residue_pairs = np.unique(residue_pairs, axis=0)
    graph = nx.Graph()
    graph.add_nodes_from(
        (i, {"subunit_id": subunit_id, "residue_id": int(r.resid), "residue_name": r.resname})
        for i, r in enumerate(residues)
    )
    graph.add_edges_from(residue_pairs.tolist())
    return graph


In [ ]:
# Import Modules
import os
import numpy as np
from numpy import savetxt
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.pylab as pl
from mpl_toolkits.mplot3d import Axes3D
import MDAnalysis as md
from MDAnalysis.analysis.distances import distance_array, contact_matrix
from MDAnalysis.analysis import contacts
from MDAnalysis.analysis.hydrogenbonds.hbond_analysis import HydrogenBondAnalysis as HBA
import ipywidgets as widgets
import py3Dmol
from tqdm import tqdm
import pandas as pd
import time

In [ ]:
prot = "lysozyme"  # lysozyme, thaumatin, apoferritin
rThresh = 6.0  # heavy-atom cutoff, angstrom

gro = str(tutorial_data(f"conf/{prot}.gro"))
traj = str(tutorial_data(f"conf/{prot}-unfolding.xtc"))
u = md.Universe(gro, traj, refresh_offsets=True)

In [ ]:
import io

heavyProtein = u.select_atoms("protein and not name H*")
with io.StringIO() as pdb_buffer:
    with md.Writer(pdb_buffer, format="PDB", multiframe=True) as writer:
        for ts in u.trajectory:
            writer.write(heavyProtein)
        trajectory_pdb = pdb_buffer.getvalue()

view = py3Dmol.view(width=600, height=400)
view.addModelsAsFrames(trajectory_pdb, "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()

def set_frame(frame):
    view.setFrame(frame)
    view.show()

widgets.interact(set_frame, frame=(0, u.trajectory.n_frames - 1))

In [ ]:
# Extract positions
frame = 0 # Set the frame you want to analyze; 0 --> unfolded, 11 --> folded
u.trajectory[frame]

# Identify protein heavy atoms
heavyProtein = u.select_atoms("protein and not name H*")
CA = u.select_atoms("protein and name CA").positions

# Store positions
heavyPos = heavyProtein.positions

# Split into atoms and residues for indexing
heavyAtoms = heavyProtein.split("atom")
heavyResidues = heavyProtein.split("residue")
heavyResids = heavyProtein.resids
n = len(heavyResidues)

### Creating a Graph Representation of a Residue Interaction Network
To construct a graph, we identify all residue-residue contacts that satisfy the following criteria:


1.   Residues *i,j* are greater than 3 residues apart in sequence space.
2.   Residues *i,j* are within a cut-off distance defined by `rThresh`.

Each residue is considered a node, with a position (irrelevant) defined by its alpha carbon. Edges are drawn between nodes that satisfy the above criteria.



In [ ]:
# Construct the same minimum-heavy-atom contact network using a spatial query.
protein = u.select_atoms("protein")
contact_graph = heavy_atom_rin(protein, cutoff=rThresh, sequence_exclusion=3)
nodes = list(contact_graph.nodes)
uniqueEdges = list(contact_graph.edges)

In [ ]:
# Construct graph
G = nx.Graph() # Initialize graph
G.add_nodes_from(nodes) # Add nodes
G.add_edges_from(uniqueEdges) # Add edges for a given cut-off

In [ ]:
# Visualize graph
import plotly.graph_objects as go

labels = [f"{heavyResidues[k].resnames[0]}{heavyResidues[k].resids[0]}" for k in range(n)]

edge_x, edge_y, edge_z = [], [], []
for i, j in G.edges():
    edge_x += [CA[i,0], CA[j,0], None]
    edge_y += [CA[i,1], CA[j,1], None]
    edge_z += [CA[i,2], CA[j,2], None]

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode='lines', line=dict(color='gray', width=2), hoverinfo='none'))
fig.add_trace(go.Scatter3d(x=CA[:,0], y=CA[:,1], z=CA[:,2], mode='markers',
                            marker=dict(size=4, color=list(range(n)), colorscale='Rainbow'),
                            text=labels, hoverinfo='text'))
fig.show()

### Centrality Measures

**Degree centrality** is the simplest measure: for each residue $i$, it's the number of contacts (edges) divided by the maximum possible.

$$C_D(i) = \frac{\deg(i)}{n-1}$$

where $\deg(i)$ is the number of edges connected to residue $i$ and $n$ is the total number of residues. Residues with high degree centrality are hubs with many simultaneous contacts — often buried core residues with dense packing.

**Closeness centrality** is the inverse of the average shortest-path distance from residue $i$ to all other residues.

$$C_C(i) = \frac{n-1}{\sum_{j \neq i} d(i,j)}$$

where $d(i,j)$ is the shortest-path distance (number of edges) between residues $i$ and $j$. Residues with high closeness can reach the rest of the structure in few steps, often corresponding to geometrically central positions in the fold.

**Betweenness centrality** quantifies how often residue $i$ lies on the shortest path between other pairs of residues.

$$C_B(i) = \sum_{s \neq i \neq t} \frac{\sigma_{st}(i)}{\sigma_{st}}$$

where $\sigma_{st}$ is the total number of shortest paths between residues $s$ and $t$, and $\sigma_{st}(i)$ is the number of those paths that pass through $i$. High betweenness residues act as bridges or bottlenecks connecting otherwise distant parts of the network — these can be mechanically or allosterically important even if not themselves highly connected (low degree).

In [ ]:
# Betweenness centrality
betweenness = nx.betweenness_centrality(G)
bc_vals = [betweenness[k] for k in range(n)]
bc_labels = [f"{labels[k]}<br>BC: {bc_vals[k]:.3f}" for k in range(n)]

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode='lines', line=dict(color='gray', width=2), hoverinfo='none'))
fig.add_trace(go.Scatter3d(x=CA[:,0], y=CA[:,1], z=CA[:,2], mode='markers',
                            marker=dict(size=4, color=bc_vals, colorscale='Magma', colorbar=dict(title="Betweenness")),
                            text=bc_labels, hoverinfo='text'))
fig.update_layout(title="Betweenness Centrality")
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
#%%
# Closeness centrality
closeness = nx.closeness_centrality(G)
cc_vals = [closeness[k] for k in range(n)]
cc_labels = [f"{labels[k]}<br>CC: {cc_vals[k]:.3f}" for k in range(n)]

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode='lines', line=dict(color='gray', width=2), hoverinfo='none'))
fig.add_trace(go.Scatter3d(x=CA[:,0], y=CA[:,1], z=CA[:,2], mode='markers',
                            marker=dict(size=4, color=cc_vals, colorscale='Magma', colorbar=dict(title="Closeness")),
                            text=cc_labels, hoverinfo='text'))
fig.update_layout(title="Closeness Centrality")
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
#%%
# Degree centrality
degree = nx.degree_centrality(G)
dc_vals = [degree[k] for k in range(n)]
dc_labels = [f"{labels[k]}<br>DC: {dc_vals[k]:.3f}" for k in range(n)]

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode='lines', line=dict(color='gray', width=2), hoverinfo='none'))
fig.add_trace(go.Scatter3d(x=CA[:,0], y=CA[:,1], z=CA[:,2], mode='markers',
                            marker=dict(size=4, color=dc_vals, colorscale='Magma', colorbar=dict(title="Degree")),
                            text=dc_labels, hoverinfo='text'))
fig.update_layout(title="Degree Centrality")
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Show centrality measurements correlation
fig, axes = plt.subplots(1, 3, figsize=(12,4))

axes[0].scatter(bc_vals, cc_vals, s=15, c="maroon")
axes[0].set_xlabel("Betweenness")
axes[0].set_ylabel("Closeness")

axes[1].scatter(cc_vals, dc_vals, s=15, c="maroon")
axes[1].set_xlabel("Closeness")
axes[1].set_ylabel("Degree")

axes[2].scatter(dc_vals, bc_vals, s=15, c="maroon")
axes[2].set_xlabel("Degree")
axes[2].set_ylabel("Betweenness")

plt.tight_layout()
plt.show()

In [ ]:
# Plot centrality measurements for all residues
fig, axes = plt.subplots(3, 1, figsize=(10,5), sharex=True)

data = [(bc_vals, "Betweenness"), (cc_vals, "Closeness"), (dc_vals, "Degree")]

for ax, (vals, name) in zip(axes, data):
    ax.plot(nodes, vals, c="black")
    ax.set_ylabel(name)

    top5 = np.argsort(vals)[-5:]
    for idx in top5:
        ax.scatter(idx, vals[idx], marker='s', c="firebrick", zorder=3)
        resname = heavyResidues[idx].resnames[0].capitalize()
        resid = heavyResidues[idx].resids[0]
        ax.annotate(f"{resname}{resid}", (idx, vals[idx]), textcoords="offset points", xytext=(0,8), ha='center', fontsize=8)
    ax.set_xlim(0, n - 1)

axes[2].set_xlabel("Residue index")

plt.tight_layout()
plt.show()

### Embedding Information on Edges

So far, edges have only encoded *whether* a contact exists. In practice, edges can carry additional information — contact strength, frequency over a trajectory, interaction energy, or any other pairwise property — as edge weights. Below, we assign synthetic weights to demonstrate how this changes the graph: weighted betweenness centrality recomputes shortest paths using these weights rather than treating every edge as equal, which can shift which residues appear as critical "bridges" in the network. I and others have used this approach to build RIN variants, such as energetic or dynamic RINs.

In [ ]:
# Reconstruct graph with synthetic edge weights and weighted betweenness centrality
G_weighted = nx.Graph()
G_weighted.add_nodes_from(nodes)

rng = np.random.default_rng(42)
for i, j in uniqueEdges:
    weight = rng.uniform(0.1, 1.0)  # synthetic edge weight (e.g., could represent contact strength, frequency, dynamic cross-correlations, etc.)
    G_weighted.add_edge(i, j, weight=weight)

# Weighted betweenness centrality (NetworkX treats 'weight' as distance by default --
# higher weight = longer path, so invert if weight should represent "strength")
bc_weighted = nx.betweenness_centrality(G_weighted, weight='weight')
bc_w_vals = [bc_weighted[k] for k in range(n)]
bc_w_labels = [f"{labels[k]}<br>BC: {bc_w_vals[k]:.3f}" for k in range(n)]

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode='lines', line=dict(color='gray', width=2), hoverinfo='none'))
fig.add_trace(go.Scatter3d(x=CA[:,0], y=CA[:,1], z=CA[:,2], mode='markers',
                            marker=dict(size=4, color=bc_w_vals, colorscale='Magma', colorbar=dict(title="Weighted Betweenness")),
                            text=bc_w_labels, hoverinfo='text'))
fig.update_layout(title="Betweenness Centrality (Synthetic Edge Weights)")
fig.update_layout(showlegend=False)
fig.show()

### Community Detection

Beyond individual residue centrality, we can ask whether the network naturally partitions into groups of densely interconnected residues — *communities*. Residues within a community tend to have more contacts with each other than with residues outside it, often corresponding to structural domains, subdomains, or cooperatively folding units. Below, we use the Louvain method (`nx.community.louvain_communities`) to partition the original (unweighted) graph and color each residue by its assigned community.

In [ ]:
# Community detection via Louvain method
communities = nx.community.louvain_communities(G, seed=42)

# Assign a community index to each node
community_id = np.zeros(n, dtype=int)
for c_idx, community in enumerate(communities):
    for node in community:
        community_id[node] = c_idx

comm_labels = [f"{labels[k]}<br>Community: {community_id[k]}" for k in range(n)]

import plotly.express as px

palette = px.colors.qualitative.Set1
node_colors = [palette[community_id[k] % len(palette)] for k in range(n)]

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode='lines', line=dict(color='gray', width=2), hoverinfo='none'))
fig.add_trace(go.Scatter3d(x=CA[:,0], y=CA[:,1], z=CA[:,2], mode='markers',
                            marker=dict(size=4, color=node_colors),
                            text=comm_labels, hoverinfo='text'))
fig.update_layout(title="Community Detection (Louvain)")
fig.update_layout(showlegend=False)
fig.show()

### Some Questions to Think About!
*   What are the highest betweenness, closeness, and degree centrality residues in the **folded** state?
*   What are the highest betweenness, closeness, and degree centrality residues in the **unfolded** state?
*   Are there any residues that overlap in terms of centrality?
*   Are there any residues that have high centrality regardless of folded state?
*   What additional properties are suitable for edge embedding?





## AAV8 capsid

One MD frame (334; 400.8 ns). Run the setup cell above, then this section: single VP → centrality RINs → complete capsid.

In [ ]:
"""Small display helpers for the accompanying static AAV8 tutorial."""
import html
import json
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display


def backbone_chains(atoms, atoms_per_vp):
    """Extract cartoon atoms from a complete GRO; VP IDs follow atom order."""
    if len(atoms) % atoms_per_vp:
        raise ValueError("Atom count does not match the recorded VP boundaries")
    chains = {}
    for vp_id, start in enumerate(range(0, len(atoms), atoms_per_vp)):
        part = atoms[start:start + atoms_per_vp]
        backbone = part.select_atoms("name N CA C O OT1")
        chains[vp_id] = dict(xyz=backbone.positions.copy(), atom_names=backbone.names,
                             resids=backbone.resids, resnames=backbone.resnames)
    return chains


def ca_nodes(vp):
    selected = vp['atom_names'] == 'CA'
    resids, names, xyz = vp['resids'][selected], vp['resnames'][selected], vp['xyz'][selected].astype(np.float64)
    if len(set(resids)) != len(resids) or not np.isfinite(xyz).all():
        raise ValueError("Ambiguous residue identities or nonfinite coordinates")
    return resids, names, xyz


def _iframe(page, height, title):
    return HTML(f'<iframe title="{html.escape(title)}" '
                f'srcdoc="{html.escape(page, quote=True)}" '
                f'style="width:100%;height:{height}px;border:0;" '
                'allowfullscreen></iframe>')


def structure_page(chains, title, full_capsid=False):
    # Each chain is its own model: serials stay below 100000, and case-sensitive
    # chain names (A/a) remain distinct even in a renderer with PDB limitations.
    alphabet = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789'
    models = []
    for vp_id, atoms in chains.items():
        lines = []
        for i, (name, resid, resname, xyz) in enumerate(zip(atoms['atom_names'], atoms['resids'], atoms['resnames'], atoms['xyz']), 1):
            x,y,z = xyz
            name = 'O' if name == 'OT1' else name  # terminal carbonyl, display alias only
            lines.append(f'ATOM  {i:5d} {name:^4s} {resname:3s} {alphabet[vp_id]}{resid:4d}    '
                         f'{x:8.3f}{y:8.3f}{z:8.3f}{1.0:6.2f}{0.0:6.2f}      CAPS{name[0]:>2s}')
        models.append('\n'.join(lines) + '\nTER\nEND\n')
    palette = ["#4c78a8", "#72b7b2", "#f2cf5b", "#f58518", "#b279a2", "#8cbf88"]
    page = """<!doctype html><html><head><meta charset="utf-8">
<style>body{margin:0;background:white;font-family:Arial,sans-serif;color:#253244}
h3{font-size:18px;font-weight:500;margin:14px 18px 0}
#viewer{position:absolute;top:44px;bottom:26px;left:0;right:0}
#status{position:absolute;bottom:5px;left:18px;font-size:12px;color:#667085}</style>
<script src="https://3Dmol.org/build/3Dmol-min.js"></script></head>
<body><h3>__TITLE__</h3><div id="viewer"></div>
<div id="status">Loading 3D view…</div><script>
try {
 if (typeof $3Dmol === 'undefined') throw new Error('3Dmol.js could not load; check your internet connection and rerun.');
 const models = __MODELS__;
 const palette = __PALETTE__;
 window.viewer = $3Dmol.createViewer('viewer', {backgroundColor:'white', antialias:true});
 models.forEach((pdb,i) => {
   const model = viewer.addModel(pdb, 'pdb');
   model.setStyle({}, {cartoon: __STYLE__});
 });
 viewer.zoomTo(); viewer.zoom(0.92); viewer.render();
 document.getElementById('status').textContent = 'Drag to rotate · Scroll to zoom';
 window.renderComplete = true;
} catch (e) { document.getElementById('status').textContent = e.message; window.renderError=String(e); }
</script></body></html>"""
    return (page.replace("__TITLE__", html.escape(title))
            .replace("__MODELS__", json.dumps(models))
            .replace("__PALETTE__", json.dumps(palette))
            .replace("__STYLE__", "{color:palette[i%palette.length]}" if full_capsid
                     else "{color:'spectrum'}"))


def show_structure(chains, title, full_capsid=False):
    display(_iframe(structure_page(chains, title, full_capsid), 540, title))


def rin_figure(graph, xyz, resids, names, values, title, chain):
    edge_xyz = np.full((graph.number_of_edges(), 3, 3), np.nan)
    for row, (i, j) in enumerate(graph.edges):
        edge_xyz[row, :2] = xyz[[i, j]]
    edge_xyz = edge_xyz.reshape(-1, 3)
    fig = go.Figure([
        go.Scatter3d(x=edge_xyz[:, 0], y=edge_xyz[:, 1], z=edge_xyz[:, 2],
                     mode="lines", line=dict(width=2, color="#bcc3cd"),
                     opacity=0.6, hoverinfo="skip", showlegend=False),
        go.Scatter3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers",
                     marker=dict(size=4, color=[values[i] for i in graph.nodes],
                                 colorscale="Magma", cmin=0,
                                 colorbar=dict(title=title, thickness=14, len=0.75)),
                     customdata=[[chain, int(resids[i]), names[i], values[i]]
                                 for i in graph.nodes],
                     hovertemplate="VP%{customdata[0]} · %{customdata[2]} %{customdata[1]}"
                                   "<br>" + title + ": %{customdata[3]:.5f}<extra></extra>",
                     showlegend=False),
    ])
    axis = dict(visible=False, showbackground=False)
    fig.update_layout(title=dict(text=f"Single VP · {title}", x=0.03), height=520,
                      margin=dict(l=0, r=0, t=45, b=0), paper_bgcolor="white",
                      font=dict(family="Arial", color="#253244"),
                      scene=dict(xaxis=axis, yaxis=axis, zaxis=axis,
                                 aspectmode="data", bgcolor="white",
                                 camera=dict(eye=dict(x=1.5, y=1.5, z=1.1))))
    return fig


def show_rin(graph, xyz, resids, names, values, title, chain):
    fig = rin_figure(graph, xyz, resids, names, values, title, chain)
    page = fig.to_html(full_html=True, include_plotlyjs="cdn",
                       config=dict(displaylogo=False, scrollZoom=True, responsive=True))
    display(_iframe(page, 535, title))


In [ ]:
VP = 0
aav8_metadata = json.loads(tutorial_data("aav8/metadata.json").read_text())
vp_universe = md.Universe(str(tutorial_data("aav8/aav8_vp0_frame334.gro")))
vp_protein = vp_universe.select_atoms("protein")
assert len(vp_protein) == aav8_metadata["single_vp_atoms"]
vp_structure = backbone_chains(vp_protein, aav8_metadata["single_vp_atoms"])
vp_resids, vp_names, vp_xyz = ca_nodes(vp_structure[VP])
assert np.array_equal(vp_resids, vp_protein.residues.resids)
print(f"VP{VP}: {len(vp_protein):,} atoms · {len(vp_resids)} residues")

### Single VP

In [ ]:
show_structure(vp_structure, f"AAV8 · single VP (VP{VP})")

### Single-VP RIN

Same as lysozyme: minimum heavy-atom distance <6 Å; exclude sequence neighbors ≤3. Exact, unweighted centralities. CA coordinates only position the nodes in the view.

In [ ]:
HEAVY_CUTOFF = 6.0  # angstrom
SEQUENCE_EXCLUSION = 3

vp_graph = heavy_atom_rin(
    vp_protein, cutoff=HEAVY_CUTOFF,
    sequence_exclusion=SEQUENCE_EXCLUSION, subunit_id=VP,
)
print(f"{vp_graph.number_of_nodes()} residues · {vp_graph.number_of_edges()} contacts")

In [ ]:
vp_degree = nx.degree_centrality(vp_graph)
show_rin(vp_graph, vp_xyz, vp_resids, vp_names, vp_degree, "Degree", VP)

In [ ]:
vp_closeness = nx.closeness_centrality(vp_graph, wf_improved=True)
show_rin(vp_graph, vp_xyz, vp_resids, vp_names, vp_closeness, "Closeness", VP)

In [ ]:
vp_betweenness = nx.betweenness_centrality(
    vp_graph, normalized=True, weight=None, endpoints=False,
)
show_rin(vp_graph, vp_xyz, vp_resids, vp_names, vp_betweenness, "Betweenness", VP)

### Complete capsid

Read the complete 60-VP GRO and display a backbone cartoon, colored by VP.

In [ ]:
capsid_universe = md.Universe(str(tutorial_data("aav8/aav8_capsid_frame334.gro")))
capsid_protein = capsid_universe.select_atoms("protein")
assert len(capsid_protein) == aav8_metadata["capsid_atoms"]
capsid = backbone_chains(capsid_protein, aav8_metadata["single_vp_atoms"])
assert len(capsid) == aav8_metadata["full_capsid_subunits"]
print(f"{len(capsid)} VPs · {len(capsid_protein.residues):,} residues · {len(capsid_protein):,} atoms")
show_structure(capsid, "AAV8 · complete capsid", full_capsid=True)